# Hour 1: Your First LangGraph Graph — The Wiring Diagram

**Goal:** Build a complete LangGraph application from scratch — no LLMs, no API keys, just pure graph mechanics.

**What you'll learn:**
- How to define **State** (the shared data structure every node reads/writes)
- How to write **Nodes** (plain Python functions that do one job)
- How to wire **Edges** (fixed connections) and **Conditional Edges** (decision points)
- How to **compile and invoke** a graph

**Prerequisites:** Python 3.10+, `pip install langgraph`

**Time:** ~45 minutes

---

## The Mental Model

A LangGraph application is a **military operations order (OPORD) turned into code.**

| Military Concept | LangGraph Equivalent | What It Does |
|---|---|---|
| Common Operational Picture (COP) | **State** (`TypedDict`) | Shared data every node reads from and writes to |
| Staff Section (S-2, S-3, etc.) | **Node** (Python function) | Does one job, returns only the fields it updates |
| Fixed arrow on flowchart | **Edge** | "After node A, always go to node B" |
| Decision diamond | **Conditional Edge** | Router function decides the next node at runtime |
| The full OPORD | **Compiled Graph** | The complete, executable application |

The key insight: **nodes don't know about each other.** The graph handles all routing. This is what makes the system composable — you can swap, add, or remove nodes without touching the others.

## The 5-Step Pattern

Every LangGraph application follows the same skeleton. Memorize this — it's your template for everything that follows:

1. **Define the State** — a `TypedDict` with the fields your graph needs
2. **Define the Nodes** — plain functions: take state in, return updates out
3. **Define the Router(s)** — functions that return the *name* of the next node
4. **Wire the Graph** — connect nodes with edges and conditional edges
5. **Compile and Run** — `.compile()` validates, `.invoke()` executes

Let's build each step.

## Step 1: Define the State

The State is your "Common Operational Picture" — the single shared data object that flows through the entire graph.

**Rules:**
- Defined as a Python `TypedDict` (a dict with a type contract)
- Every node receives the **full** state as input
- Every node returns **only the fields it wants to update**
- Fields not included in the return value stay unchanged

> **Think of it this way:** The state is a whiteboard in the operations center. Any staff section can read it, and any section can update their part of it. But they don't erase what other sections wrote.

In [ ]:
from typing import TypedDict
from datetime import datetime
from langgraph.graph import StateGraph, START, END

# ─── STEP 1: DEFINE THE STATE ───────────────────────────────────────────
# This is the "COP" — shared data structure for the entire graph.
# TypedDict gives you type hints (IDE autocomplete) while still being a regular dict at runtime.

class IntelState(TypedDict):
    report: str          # Input: the raw intelligence report text
    threat_level: str    # Set by analyze_intel: "HIGH", "MEDIUM", or "LOW"
    action: str          # Set by respond_* nodes: what we're doing about it
    timestamp: str       # Set by analyze_intel: when the analysis happened

# Quick check — it's still just a dict under the hood:
sample_state: IntelState = {
    "report": "Test report",
    "threat_level": "",
    "action": "",
    "timestamp": ""
}
print(f"State type: {type(sample_state)}")
print(f"State keys: {list(sample_state.keys())}"

## Step 2: Define the Nodes

Each node is a **plain Python function**. No special base class, no decorators, no magic.

**Node contract:**
- Takes ONE argument: the full state (typed as your `TypedDict`)
- Returns a `dict` with ONLY the fields to update
- Should NOT mutate the input state — return new values
- Should NOT know about other nodes — the graph handles routing

> **Why this matters for your portfolio:** This isolation is what makes LangGraph production-ready. You can swap any node independently (e.g., replace keyword matching with an LLM call in Hour 3), test each node in isolation (just pass it a dict), and reuse nodes across different graphs.

In [ ]:
# ─── STEP 2: DEFINE THE NODES ───────────────────────────────────────────
# Each node: takes state → does one thing → returns updates

def analyze_intel(state: IntelState) -> dict:
    """
    NODE: analyze_intel
    Role: S-2 Intelligence — assess the threat level from a raw report.
    Reads: report
    Writes: threat_level, timestamp
    
    Right now this is simple keyword matching.
    In Hour 3, you'll replace this with an LLM call — 
    and the rest of the graph won't change at all.
    """
    report = state["report"].lower()
    now = datetime.now().isoformat()

    # Classify threat based on keywords
    if "enemy" in report or "hostile" in report:
        threat = "HIGH"
    elif "suspicious" in report or "unusual" in report:
        threat = "MEDIUM"
    else:
        threat = "LOW"

    # Return ONLY the fields we're updating.
    # "report" stays unchanged because we don't include it here.
    return {"threat_level": threat, "timestamp": now}


def respond_high(state: IntelState) -> dict:
    """NODE: respond_high — Execute HIGH threat response."""
    return {"action": f"ENGAGE: Mobilizing QRF for report: {state['report']}"}


def respond_medium(state: IntelState) -> dict:
    """NODE: respond_medium — Execute MEDIUM threat response."""
    return {"action": f"INVESTIGATE: Dispatching recon element. Report: {state['report']}"}


def respond_low(state: IntelState) -> dict:
    """NODE: respond_low — Execute LOW threat response."""
    return {"action": f"MONITOR: Continuing patrol. Report: {state['report']}"}


# Quick test — nodes are just functions, so you can call them directly:
test_state = {"report": "Enemy convoy spotted", "threat_level": "", "action": "", "timestamp": ""}
result = analyze_intel(test_state)
print(f"analyze_intel returned: {result}")
print(f"Notice: only threat_level and timestamp — 'report' and 'action' are untouched"

## Step 3: Define the Router

A router is a function used by a **Conditional Edge**. It reads the current state and returns a **string** — the name of the next node.

**Critical rules:**
- The returned string MUST exactly match a node name registered with `add_node()`
- LangGraph won't warn you about typos — it'll error at runtime
- Every possible return value must map to a registered node

> **This is the "decision diamond" in your flowchart.** In Hour 1, the router uses `if/else`. In Hour 3, the router will use an LLM to make the decision. The graph topology stays identical either way.

In [ ]:
# ─── STEP 3: DEFINE THE ROUTER ──────────────────────────────────────────
# The router reads state and returns the NAME of the next node as a string.
# This function IS the "decision diamond" in the flowchart.

def route_by_threat(state: IntelState) -> str:
    """
    ROUTER: Decides which respond_* node to run next.
    Returns a string that MUST match a registered node name.
    """
    if state["threat_level"] == "HIGH":
        return "respond_high"       # ← Must match add_node("respond_high", ...)
    elif state["threat_level"] == "MEDIUM":
        return "respond_medium"     # ← Must match add_node("respond_medium", ...)
    return "respond_low"            # ← Default / fallback path

# Test the router directly:
print(f"HIGH threat routes to: {route_by_threat({'report': '', 'threat_level': 'HIGH', 'action': '', 'timestamp': ''})}")
print(f"LOW threat routes to:  {route_by_threat({'report': '', 'threat_level': 'LOW', 'action': '', 'timestamp': ''})}"

## Step 4: Wire the Graph

Now we assemble the OPORD. This is where you connect nodes with edges.

**Three types of connections:**
- `add_edge(START, "node")` — fixed entry point
- `add_edge("node_a", "node_b")` — fixed connection (always go from A to B)
- `add_conditional_edges("node", router_fn, path_map)` — dynamic routing based on state

The path map (`{router_return_value: target_node}`) is technically optional, but being explicit is best practice — it documents your graph and catches typos.

In [ ]:
# ─── STEP 4: WIRE THE GRAPH ─────────────────────────────────────────────
graph = StateGraph(IntelState)

# Register nodes — order doesn't matter, just like org chart slots
graph.add_node("analyze_intel", analyze_intel)
graph.add_node("respond_high", respond_high)
graph.add_node("respond_medium", respond_medium)
graph.add_node("respond_low", respond_low)

# FIXED EDGE: START → analyze_intel
# "No matter what, the first thing we do is analyze the intel."
graph.add_edge(START, "analyze_intel")

# CONDITIONAL EDGE: analyze_intel → ??? (depends on router output)
# The path map makes the routing explicit and self-documenting.
graph.add_conditional_edges(
    "analyze_intel",         # Source node
    route_by_threat,         # Router function
    {                        # Path map: {router_return → target_node}
        "respond_high": "respond_high",
        "respond_medium": "respond_medium",
        "respond_low": "respond_low",
    },
)

# FIXED EDGES: all response nodes → END
graph.add_edge("respond_high", END)
graph.add_edge("respond_medium", END)
graph.add_edge("respond_low", END)

print("Graph wired successfully!")
print(f"Nodes: {list(graph.nodes.keys())}"

## Step 5: Compile and Run

`.compile()` validates the graph (checks for missing edges, unreachable nodes) and returns a runnable object. After compiling, you can't add more nodes or edges.

`.invoke()` runs the graph synchronously — pass in the initial state, get back the final state after all nodes have executed.

Let's test all three threat paths:

In [ ]:
# ─── STEP 5: COMPILE AND RUN ────────────────────────────────────────────
app = graph.compile()

# TEST 1: HIGH threat
print("=" * 60)
print("TEST 1: HIGH threat")
print("=" * 60)
result1 = app.invoke({
    "report": "Enemy convoy spotted on MSR Tampa",
    "threat_level": "",
    "action": "",
    "timestamp": "",
})
print(f"  Threat:    {result1['threat_level']}")
print(f"  Action:    {result1['action']}")
print(f"  Timestamp: {result1['timestamp']}")

print()

# TEST 2: MEDIUM threat
print("=" * 60)
print("TEST 2: MEDIUM threat")
print("=" * 60)
result2 = app.invoke({
    "report": "Suspicious activity near the bridge",
    "threat_level": "",
    "action": "",
    "timestamp": "",
})
print(f"  Threat:    {result2['threat_level']}")
print(f"  Action:    {result2['action']}")

print()

# TEST 3: LOW threat
print("=" * 60)
print("TEST 3: LOW threat")
print("=" * 60)
result3 = app.invoke({
    "report": "Routine traffic on Highway 1",
    "threat_level": "",
    "action": "",
    "timestamp": "",
})
print(f"  Threat:    {result3['threat_level']}")
print(f"  Action:    {result3['action']}"

## Key Takeaways

**What you just built:** A complete LangGraph application with conditional routing — the same topology used in production multi-agent systems.

**The 5-step pattern works for ANY graph:**
1. Define the State (`TypedDict`)
2. Define the Nodes (plain functions)
3. Define the Router(s) (return node names as strings)
4. Wire the Graph (edges + conditional edges)
5. Compile and Run (`.compile()` → `.invoke()`)

**What's missing (and what Hours 2-4 add):**
- **Hour 2:** State reducers (accumulate instead of overwrite), loops, checkpointing
- **Hour 3:** LLMs inside nodes, message-based state, the supervisor pattern
- **Hour 4:** Production RAG pipeline with quality gates and feedback loops

> **Portfolio signal:** In an interview, you can articulate that the graph topology is *separate* from node implementation. The same wiring works whether nodes use keyword matching or LLM calls. That separation is the architectural insight.

---
*Next: [02_research_pipeline.ipynb](./02_research_pipeline.ipynb) — State reducers, loops, and checkpointing*